# 7-MLET Datathon: Contextual Bandits for Financial Offers

This notebook builds a complete, reproducible adaptive experimentation pipeline for marketing offers.

What you will get:
- notebook-local dependency validation with `sys.executable -m pip`
- Azure ML workspace configuration and URI_FOLDER data asset registration
- Feature Store-ready curated features, MLTable metadata, and agentic feature discovery
- segregated utility notebooks for environment, ingestion, feature store, and bandit execution
- Foundry agent binding to the `rg-microsoft-iq` project and the deployed `finance-orchestrator` agent
- Azure ML Feature Store artifacts with agent-reviewed feature propositions
- staged Kaggle/local/OpenML ingestion into a local folder before loading data
- preprocessing with explicit leakage handling (`duration` removed)
- synthetic contextual bandit environment with delayed rewards
- deterministic baseline, UCB, and Thompson Sampling policies
- aggregate-only Foundry agent validation after the bandit metrics are produced


## 1) Notebook Dependency Bootstrap

Validate the packages this notebook needs before importing the pipeline dependencies.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "azure-ai-ml": "azure.ai.ml",
    "azure-ai-projects": "azure.ai.projects",
    "azure-identity": "azure.identity",
    "python-dotenv": "dotenv",
    "kaggle": "kaggle",
    "mltable": "mltable",
}

def module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

missing_packages = [
    package_name
    for package_name, module_name in REQUIRED_PACKAGES.items()
    if not module_available(module_name)
]

if missing_packages:
    print("Installing missing packages with the active notebook interpreter:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing_packages,
    ])
    importlib.invalidate_caches()
else:
    print("All required packages are already available in this notebook environment.")

print("Python executable:", sys.executable)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import fetch_openml

candidate_src_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "aml-foundry-integration" / "src",
    Path.cwd().parent.parent / "aml-foundry-integration" / "src",
]
for package_src in candidate_src_paths:
    if (package_src / "aml_bandits" / "settings.py").exists():
        if str(package_src) not in sys.path:
            sys.path.insert(0, str(package_src))
        break

RNG = np.random.default_rng(42)
pd.set_option("display.max_columns", 120)


## 2) Azure ML Configuration

Use environment variables first. The resource group and workspace defaults match this local demo workspace and do not include secrets.

In [ ]:
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import DefaultAzureCredential

from aml_bandits.settings import (
    apply_project_environment_defaults,
    load_project_defaults,
    project_summary,
)

load_dotenv()
PROJECT_DEFAULTS = apply_project_environment_defaults(load_project_defaults())
PROJECT_BINDING = project_summary(PROJECT_DEFAULTS)

DEFAULT_AZUREML_RESOURCE_GROUP = PROJECT_DEFAULTS.azure_ml.resource_group
DEFAULT_AZUREML_WORKSPACE_NAME = PROJECT_DEFAULTS.azure_ml.workspace_name
DEFAULT_AZUREML_LOCATION = PROJECT_DEFAULTS.azure_ml.location

AZUREML_SUBSCRIPTION_ID = (
    os.getenv("AZUREML_SUBSCRIPTION_ID")
    or os.getenv("AZURE_SUBSCRIPTION_ID")
    or os.getenv("SUBSCRIPTION_ID")
    or ""
)
AZUREML_RESOURCE_GROUP = os.getenv("AZUREML_RESOURCE_GROUP", DEFAULT_AZUREML_RESOURCE_GROUP)
AZUREML_WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", DEFAULT_AZUREML_WORKSPACE_NAME)
AZUREML_LOCATION = os.getenv("AZUREML_LOCATION", DEFAULT_AZUREML_LOCATION)
AZUREML_DATA_ASSET_NAME = os.getenv("AZUREML_DATA_ASSET_NAME", "bank-marketing-7mlet")
AZUREML_DATA_ASSET_VERSION = os.getenv("AZUREML_DATA_ASSET_VERSION", "1")

INGESTION_DIR = Path(
    os.getenv(
        "BANK_MARKETING_INGESTION_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "azureml_ingestion"),
    )
).resolve()
INGESTION_DIR.mkdir(parents=True, exist_ok=True)
os.environ["BANK_MARKETING_INGESTION_DIR"] = str(INGESTION_DIR)

AZUREML_SPEC_DIR = Path(
    os.getenv(
        "AZUREML_SPEC_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "azureml_specs"),
    )
).resolve()
AZUREML_SPEC_DIR.mkdir(parents=True, exist_ok=True)

print("Azure ML resource group:", AZUREML_RESOURCE_GROUP)
print("Azure ML workspace:", AZUREML_WORKSPACE_NAME)
print("Azure ML location:", AZUREML_LOCATION)
print("Subscription configured:", bool(AZUREML_SUBSCRIPTION_ID))
print("Foundry project endpoint:", os.getenv("FOUNDRY_PROJECT_ENDPOINT"))
print("Foundry feature agent:", os.getenv("FOUNDRY_FEATURE_AGENT_NAME"))
print("Foundry strategy agent:", os.getenv("FOUNDRY_STRATEGY_AGENT_NAME"))
print("Available Foundry agents:", os.getenv("FOUNDRY_AVAILABLE_AGENTS"))
print("Local ingestion folder:", INGESTION_DIR)
print("Azure ML spec folder:", AZUREML_SPEC_DIR)


## 3) Materialize Source Data for Ingestion

Kaggle, local CSVs, or OpenML are copied or written into one local ingestion folder before the pipeline reads any rows.

In [ ]:
def _try_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists() or path.suffix.lower() != ".csv":
        return None
    for sep in [",", ";", "\t", "|"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 2:
                return df
        except Exception:
            continue
    return None


def _find_target_column(columns: Sequence[str]) -> Optional[str]:
    lowered = {c.lower(): c for c in columns}
    candidates = [
        "y",
        "target",
        "label",
        "subscribed",
        "conversion",
        "converted",
        "response",
        "class",
    ]
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    return None


def _csv_files(root: Path) -> List[Path]:
    if root.is_file() and root.suffix.lower() == ".csv":
        return [root]
    if not root.exists():
        return []
    return sorted(path for path in root.rglob("*.csv") if path.is_file())


def _download_kaggle_csvs(destination: Path) -> List[Path]:
    datasets = [
        "henriqueyamahata/bank-marketing",
        "tunguz/bank-marketing-data-set",
        "dharmik34/bank-term-deposit-subscription",
    ]
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
    except Exception as ex:
        print(f"Kaggle materialization skipped: {ex}")
        return []

    for dataset_name in datasets:
        try:
            api.dataset_download_files(
                dataset_name,
                path=str(destination),
                unzip=True,
                quiet=True,
            )
            csvs = _csv_files(destination)
            if csvs:
                return csvs
        except Exception as ex:
            print(f"Kaggle dataset skipped ({dataset_name}): {ex}")
    return []


def _copy_local_csvs(repo_root: Path, destination: Path) -> List[Path]:
    candidates = [
        repo_root / "tmp" / "7mlet" / "bank-additional-full.csv",
        repo_root / "tmp" / "7mlet" / "bank-full.csv",
        repo_root / "tmp" / "7mlet" / "bank.csv",
        repo_root / "data" / "assets" / "bank-additional-full.csv",
        repo_root / "data" / "assets" / "bank-full.csv",
    ]

    copied: List[Path] = []
    for source in candidates:
        if not source.exists():
            continue
        target = destination / source.name
        if source.resolve() != target.resolve():
            shutil.copy2(source, target)
        copied.append(target)
    return copied


def _materialize_openml_csv(destination: Path) -> List[Path]:
    try:
        bunch = fetch_openml(name="bank-marketing", version=1, as_frame=True, parser="auto")
    except TypeError:
        bunch = fetch_openml(name="bank-marketing", version=1, as_frame=True)

    output_path = destination / "bank_marketing_openml.csv"
    bunch.frame.copy().to_csv(output_path, index=False)
    return [output_path]


def materialize_bank_marketing_csvs() -> Tuple[List[Path], str]:
    repo_root = Path.cwd()
    INGESTION_DIR.mkdir(parents=True, exist_ok=True)

    existing_csvs = _csv_files(INGESTION_DIR)
    if existing_csvs:
        return existing_csvs, "existing local ingestion folder"

    kaggle_csvs = _download_kaggle_csvs(INGESTION_DIR)
    if kaggle_csvs:
        return kaggle_csvs, "Kaggle API materialized to local ingestion folder"

    local_csvs = _copy_local_csvs(repo_root, INGESTION_DIR)
    if local_csvs:
        return local_csvs, "local CSVs copied to ingestion folder"

    openml_csvs = _materialize_openml_csv(INGESTION_DIR)
    return openml_csvs, "OpenML materialized to local ingestion folder"


materialized_csvs, materialization_provenance = materialize_bank_marketing_csvs()
print(f"Materialized source: {materialization_provenance}")
print(f"Ingestion folder: {INGESTION_DIR}")
print("CSV files:", [path.name for path in materialized_csvs])

## 4) Prepare Azure ML Data Sources and Data Imports

Write a source manifest and Azure ML data import specification before registering assets. The data import command is generated every time and only executed when `ENABLE_AZUREML_DATA_IMPORT=true`.


In [ ]:
DATA_SOURCE_SPEC_DIR = AZUREML_SPEC_DIR / "data_sources"
DATA_SOURCE_SPEC_DIR.mkdir(parents=True, exist_ok=True)

ENABLE_AZUREML_DATA_IMPORT = os.getenv("ENABLE_AZUREML_DATA_IMPORT", "false").strip().lower() in {"1", "true", "yes", "y"}
RAW_DATA_SOURCE_NAME = os.getenv("AZUREML_RAW_DATA_SOURCE_NAME", f"{AZUREML_DATA_ASSET_NAME}-source")

source_manifest = {
    "name": RAW_DATA_SOURCE_NAME,
    "source_type": materialization_provenance,
    "local_landing_folder": str(INGESTION_DIR),
    "csv_files": [path.name for path in materialized_csvs],
    "azure_ml": {
        "resource_group": AZUREML_RESOURCE_GROUP,
        "workspace_name": AZUREML_WORKSPACE_NAME,
        "data_asset_name": AZUREML_DATA_ASSET_NAME,
        "data_asset_version": AZUREML_DATA_ASSET_VERSION,
    },
    "foundry_project": PROJECT_BINDING["foundry"],
    "privacy": "raw_rows_stay_in_azure_ml_data_layer",
}

source_manifest_path = DATA_SOURCE_SPEC_DIR / "raw_data_source_manifest.json"
source_manifest_path.write_text(
    json.dumps(source_manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

raw_data_import_spec_path = DATA_SOURCE_SPEC_DIR / "raw_data_import.yml"
raw_data_import_spec_path.write_text(
    "\n".join(
        [
            "$schema: https://azuremlschemas.azureedge.net/latest/data.schema.json",
            f"name: {AZUREML_DATA_ASSET_NAME}",
            f"version: {AZUREML_DATA_ASSET_VERSION}",
            "type: uri_folder",
            "description: 7-MLET raw bank marketing source imported into Azure ML from the notebook landing folder.",
            "tags:",
            "  stage: raw_ingestion",
            "  source_manifest: raw_data_source_manifest.json",
            "  foundry_project: miq-project-miqsec26",
            "  default_agent: finance-orchestrator",
            f"path: {INGESTION_DIR.as_posix()}",
            "",
        ]
    ),
    encoding="utf-8",
)

data_import_command = [
    "az",
    "ml",
    "data",
    "import",
    "--file",
    str(raw_data_import_spec_path),
    "--resource-group",
    AZUREML_RESOURCE_GROUP,
    "--workspace-name",
    AZUREML_WORKSPACE_NAME,
    "--skip-validation",
]

print("Data source manifest:", source_manifest_path)
print("Data import spec:", raw_data_import_spec_path)
print("Data import command:", " ".join(data_import_command))

if ENABLE_AZUREML_DATA_IMPORT:
    completed = subprocess.run(data_import_command, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        print("Data import did not complete; SDK data asset registration remains the fallback path.")
else:
    print("Data import execution is gated. Set ENABLE_AZUREML_DATA_IMPORT=true to materialize through az ml data import.")


## 5) Register Azure ML Data Asset

Create or update a URI_FOLDER data asset from the local ingestion folder. This is separate from later Foundry agent validation, and raw customer rows are not sent to agents.

In [ ]:
from datetime import datetime, timezone

def build_ml_client() -> Optional[MLClient]:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    if AZUREML_SUBSCRIPTION_ID:
        return MLClient(
            credential=credential,
            subscription_id=AZUREML_SUBSCRIPTION_ID,
            resource_group_name=AZUREML_RESOURCE_GROUP,
            workspace_name=AZUREML_WORKSPACE_NAME,
        )

    try:
        return MLClient.from_config(credential=credential)
    except Exception as ex:
        print("Azure ML client not created. Set AZUREML_SUBSCRIPTION_ID or provide an Azure ML config file.")
        print(f"Details: {ex}")
        return None


def create_or_update_data_asset(ml_client: MLClient, data_folder: Path) -> Data:
    def data_asset(version: str) -> Data:
        return Data(
            name=AZUREML_DATA_ASSET_NAME,
            version=version,
            type=AssetTypes.URI_FOLDER,
            path=str(data_folder),
            description="7-MLET bank marketing ingestion folder for contextual bandit demo.",
        )

    try:
        return ml_client.data.create_or_update(data_asset(AZUREML_DATA_ASSET_VERSION))
    except Exception as ex:
        fallback_version = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
        print(f"Data asset version {AZUREML_DATA_ASSET_VERSION!r} was not updated: {ex}")
        print(f"Retrying with generated version {fallback_version}.")
        return ml_client.data.create_or_update(data_asset(fallback_version))


ml_client = build_ml_client()
registered_data_asset = None

if ml_client is not None:
    registered_data_asset = create_or_update_data_asset(ml_client, INGESTION_DIR)
    os.environ["AZUREML_REGISTERED_DATA_ASSET_ID"] = registered_data_asset.id
    print("Registered Azure ML data asset:", registered_data_asset.name)
    print("Version:", registered_data_asset.version)
    print("Asset id:", registered_data_asset.id)
else:
    print("Continuing with the local ingestion folder:", INGESTION_DIR)

print("For Azure ML jobs, mount or download this asset and set AZUREML_DATA_ASSET_PATH to the local mount path.")

## 6) Load Data From the Ingestion Folder

The loader prefers `AZUREML_DATA_ASSET_PATH` when Azure ML provides a local mount, then the registered local ingestion folder, then legacy local CSVs and OpenML.

In [3]:
def _local_path_from_env(var_name: str) -> Optional[Path]:
    raw_value = os.getenv(var_name, "").strip()
    if not raw_value:
        return None
    if raw_value.startswith(("azureml://", "abfss://", "wasbs://")):
        print(f"{var_name} points to a remote URI. Use a mounted local path for direct notebook loading.")
        return None
    path = Path(raw_value).expanduser().resolve()
    if not path.exists():
        print(f"{var_name} is set but does not exist locally: {path}")
        return None
    return path


def _legacy_local_paths(repo_root: Path) -> List[Path]:
    return [
        repo_root / "tmp" / "7mlet" / "bank-additional-full.csv",
        repo_root / "tmp" / "7mlet" / "bank-full.csv",
        repo_root / "tmp" / "7mlet" / "bank.csv",
        repo_root / "data" / "assets" / "bank-additional-full.csv",
        repo_root / "data" / "assets" / "bank-full.csv",
    ]


def _load_first_valid_csv(candidates: List[Tuple[Path, str]]) -> Optional[Tuple[pd.DataFrame, str, str]]:
    for root, label in candidates:
        for path in _csv_files(root):
            df = _try_read_csv(path)
            if df is None:
                continue
            target_col = _find_target_column(df.columns)
            if target_col is None:
                continue
            return df, target_col, f"{label}: {path.as_posix()}"
    return None


def load_bank_marketing_dataset() -> Tuple[pd.DataFrame, str, str]:
    repo_root = Path.cwd()
    candidate_roots: List[Tuple[Path, str]] = []

    azureml_data_asset_path = _local_path_from_env("AZUREML_DATA_ASSET_PATH")
    if azureml_data_asset_path is not None:
        candidate_roots.append((azureml_data_asset_path, "AZUREML_DATA_ASSET_PATH"))

    ingestion_path = Path(os.getenv("BANK_MARKETING_INGESTION_DIR", str(INGESTION_DIR))).resolve()
    candidate_roots.append((ingestion_path, "local Azure ML ingestion folder"))
    candidate_roots.extend((path, "legacy local CSV") for path in _legacy_local_paths(repo_root))

    prepared_data = _load_first_valid_csv(candidate_roots)
    if prepared_data is not None:
        return prepared_data

    openml_csvs = _materialize_openml_csv(INGESTION_DIR)
    prepared_data = _load_first_valid_csv([(path, "OpenML materialized fallback") for path in openml_csvs])
    if prepared_data is not None:
        return prepared_data

    raise ValueError("Unable to identify a valid bank marketing dataset with a target column.")


raw_df, target_col, provenance = load_bank_marketing_dataset()
print(f"Loaded from: {provenance}")
print(f"Shape: {raw_df.shape}")
print(f"Target column: {target_col}")
raw_df.head(3)

NameError: name 'Path' is not defined

## 7) Preprocessing and Leakage Control

Classical ML choice: for binary conversion prediction and contextual feature extraction, we combine:
- `SimpleImputer` for robust missing-value handling
- `OneHotEncoder` for categorical variables
- `StandardScaler` for numeric variables

Leakage handling: we explicitly remove `duration`, because call duration is not known before decision time.

In [ ]:
def normalize_target(series: pd.Series) -> pd.Series:
    values = series.astype(str).str.lower().str.strip()
    positive = {"yes", "1", "true", "y", "converted", "success"}
    return values.map(lambda v: 1 if v in positive else 0).astype(int)


df = raw_df.copy()
if "duration" in df.columns:
    df = df.drop(columns=["duration"])

y = normalize_target(df[target_col])
X = df.drop(columns=[target_col])

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ],
    remainder="drop",
)

print(f"Features after leakage handling: {X.shape[1]}")
print(f"Numeric columns: {len(numeric_cols)}, Categorical columns: {len(categorical_cols)}")

## 8) Agentic Feature Discovery and Feature Store Artifacts

This section turns the cleaned decision-time dataset into feature-store-ready artifacts:

- aggregate-only feature evidence for agent review
- deterministic local fallback when Foundry is not configured
- curated feature table plus MLTable metadata
- Azure ML data assets for the feature table and feature report bundle
- optional first-class Azure ML Feature Store specs and creation commands

Raw customer rows stay in the Azure ML/local data layer. The agent sees schema and aggregate statistics only.


In [ ]:
from dataclasses import asdict
import json
from typing import Any
import hashlib

FEATURE_ARTIFACT_DIR = Path(
    os.getenv(
        "BANK_MARKETING_FEATURE_ARTIFACT_DIR",
        str(Path.cwd() / "tmp" / "7mlet" / "feature_artifacts"),
    )
).resolve()
FEATURE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["BANK_MARKETING_FEATURE_ARTIFACT_DIR"] = str(FEATURE_ARTIFACT_DIR)

candidate_src_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "aml-foundry-integration" / "src",
    Path.cwd().parent.parent / "aml-foundry-integration" / "src",
]
for package_src in candidate_src_paths:
    if (package_src / "aml_bandits" / "foundry_bridge.py").exists():
        if str(package_src) not in sys.path:
            sys.path.insert(0, str(package_src))
        break

from aml_bandits.foundry_bridge import (
    build_feature_discovery_payload,
    validate_and_propose_features,
)


def _category_token(value: Any) -> str:
    text = "__MISSING__" if pd.isna(value) else str(value)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]


def _safe_float(value: Any) -> float | None:
    if value is None or pd.isna(value):
        return None
    numeric_value = float(value)
    if not np.isfinite(numeric_value):
        return None
    return numeric_value


def build_aggregate_feature_profile(
    features_frame: pd.DataFrame,
    target: pd.Series,
    target_name: str,
) -> dict[str, Any]:
    target_values = target.reset_index(drop=True).astype(int)
    feature_records: list[dict[str, Any]] = []

    for column_name in features_frame.columns:
        series = features_frame[column_name].reset_index(drop=True)
        record: dict[str, Any] = {
            "name": column_name,
            "dtype": str(series.dtype),
            "missing_rate": float(series.isna().mean()),
            "unique_count": int(series.nunique(dropna=True)),
            "decision_time_available": column_name.lower() != "duration",
            "leakage_risk": column_name.lower() == "duration",
        }

        if column_name in numeric_cols:
            numeric_values = pd.to_numeric(series, errors="coerce")
            quantiles = numeric_values.quantile([0.0, 0.25, 0.5, 0.75, 1.0]).to_dict()
            valid = pd.DataFrame({"value": numeric_values, "target": target_values}).dropna()
            target_correlation = None
            if valid["value"].nunique() > 1 and valid["target"].nunique() > 1:
                target_correlation = valid["value"].corr(valid["target"])
            record.update(
                {
                    "kind": "numeric",
                    "mean": _safe_float(numeric_values.mean()),
                    "std": _safe_float(numeric_values.std()),
                    "quantiles": {str(key): _safe_float(value) for key, value in quantiles.items()},
                    "target_correlation": _safe_float(target_correlation),
                }
            )
        else:
            tokens = series.map(_category_token)
            token_stats = (
                pd.DataFrame({"token": tokens, "target": target_values})
                .groupby("token", as_index=False)
                .agg(count=("target", "size"), positive_rate=("target", "mean"))
                .sort_values(by="count", ascending=False)
            )
            positive_rates = token_stats["positive_rate"]
            record.update(
                {
                    "kind": "categorical",
                    "top_hashed_tokens": token_stats.head(8).to_dict(orient="records"),
                    "max_positive_rate_delta": _safe_float(
                        positive_rates.max() - positive_rates.min()
                        if len(positive_rates) > 1
                        else 0.0
                    ),
                }
            )
        feature_records.append(record)

    return {
        "privacy": "aggregate_evidence_only_no_raw_customer_rows",
        "row_count": int(len(features_frame)),
        "feature_count": int(features_frame.shape[1]),
        "target_name": target_name,
        "target_positive_rate": float(target_values.mean()),
        "excluded_columns": ["duration"],
        "feature_names": features_frame.columns.tolist(),
        "features": feature_records,
    }


feature_profile = build_aggregate_feature_profile(X, y, target_col)
feature_discovery_payload = build_feature_discovery_payload(feature_profile)
feature_discovery_response = validate_and_propose_features(feature_profile)
feature_discovery_response_dict = asdict(feature_discovery_response)

print("Feature payload privacy mode:", feature_discovery_payload["privacy"])
print("Feature agent source:", feature_discovery_response.source)
print("Feature validation:", feature_discovery_response.validation)
print("Accepted base features:", len(feature_discovery_response.accepted_features))
print("Proposed derived features:", [item.get("name") for item in feature_discovery_response.proposed_features])

pd.DataFrame(
    [
        {
            "source": feature_discovery_response.source,
            "validation": feature_discovery_response.validation,
            "accepted_features": len(feature_discovery_response.accepted_features),
            "rejected_features": len(feature_discovery_response.rejected_features),
            "proposed_features": len(feature_discovery_response.proposed_features),
            "risk_flags": "; ".join(feature_discovery_response.risk_flags),
            "warnings": "; ".join(feature_discovery_response.warnings),
        }
    ]
)


In [ ]:
def build_curated_feature_table(
    features_frame: pd.DataFrame,
    response: Any,
) -> pd.DataFrame:
    accepted_base = [
        column_name
        for column_name in features_frame.columns
        if column_name in set(response.accepted_features)
    ]
    if not accepted_base:
        accepted_base = [column_name for column_name in features_frame.columns if column_name.lower() != "duration"]

    curated = features_frame[accepted_base].reset_index(drop=True).copy()
    proposal_names = {str(item.get("name")) for item in response.proposed_features}

    if "age_band" in proposal_names and "age" in features_frame.columns:
        curated["age_band"] = pd.cut(
            pd.to_numeric(features_frame["age"], errors="coerce"),
            bins=[0, 30, 45, 60, 200],
            labels=["under_30", "30_44", "45_59", "60_plus"],
        ).astype("string")

    if "campaign_pressure" in proposal_names and {"campaign", "previous"}.issubset(features_frame.columns):
        curated["campaign_pressure"] = (
            pd.to_numeric(features_frame["campaign"], errors="coerce").fillna(0)
            + pd.to_numeric(features_frame["previous"], errors="coerce").fillna(0)
        )

    if "has_prior_contact" in proposal_names:
        prior_contact = pd.Series(False, index=features_frame.index)
        if "previous" in features_frame.columns:
            prior_contact = prior_contact | (
                pd.to_numeric(features_frame["previous"], errors="coerce").fillna(0) > 0
            )
        if "pdays" in features_frame.columns:
            pdays = pd.to_numeric(features_frame["pdays"], errors="coerce")
            prior_contact = prior_contact | (~pdays.isin([-1, 999]) & pdays.notna())
        curated["has_prior_contact"] = prior_contact.astype(int)

    if "balance_per_campaign" in proposal_names and {"balance", "campaign"}.issubset(features_frame.columns):
        balance = pd.to_numeric(features_frame["balance"], errors="coerce").fillna(0)
        campaign = pd.to_numeric(features_frame["campaign"], errors="coerce").fillna(0).clip(lower=1)
        curated["balance_per_campaign"] = balance / campaign

    event_time = pd.date_range(
        start=pd.Timestamp("2024-01-01T00:00:00Z"),
        periods=len(curated),
        freq="min",
    )
    curated.insert(0, "feature_row_id", np.arange(len(curated), dtype=int))
    curated.insert(1, "event_time", event_time.astype(str))
    return curated


def write_dataframe_artifact(frame: pd.DataFrame, stem: str) -> Path:
    parquet_path = FEATURE_ARTIFACT_DIR / f"{stem}.parquet"
    try:
        frame.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as ex:
        csv_path = FEATURE_ARTIFACT_DIR / f"{stem}.csv"
        frame.to_csv(csv_path, index=False)
        print(f"Parquet unavailable ({ex}); wrote CSV artifact instead: {csv_path.name}")
        return csv_path


def write_mltable_file(data_path: Path) -> Path:
    relative_name = data_path.name
    if data_path.suffix.lower() == ".parquet":
        transformations = "  - read_parquet: {}\n"
    else:
        transformations = "  - read_delimited:\n      delimiter: ','\n      encoding: utf8\n"
    content = f"paths:\n  - file: ./{relative_name}\ntransformations:\n{transformations}"
    mltable_path = FEATURE_ARTIFACT_DIR / "MLTable"
    mltable_path.write_text(content, encoding="utf-8")
    return mltable_path


feature_store_table = build_curated_feature_table(X, feature_discovery_response)
feature_table_path = write_dataframe_artifact(feature_store_table, "bank_marketing_feature_table")
mltable_path = write_mltable_file(feature_table_path)

feature_schema = {
    "entity_name": "bank_marketing_customer",
    "index_columns": [{"name": "feature_row_id", "type": "integer"}],
    "timestamp_column": "event_time",
    "target_name": target_col,
    "source_data_asset_id": getattr(registered_data_asset, "id", None),
    "feature_table_path": feature_table_path.name,
    "mltable_path": mltable_path.name,
    "features": [
        {
            "name": column_name,
            "dtype": str(feature_store_table[column_name].dtype),
            "role": "index" if column_name == "feature_row_id" else "timestamp" if column_name == "event_time" else "feature",
        }
        for column_name in feature_store_table.columns
    ],
}

(FEATURE_ARTIFACT_DIR / "feature_profile_payload.json").write_text(
    json.dumps(feature_discovery_payload, indent=2, sort_keys=True),
    encoding="utf-8",
)
(FEATURE_ARTIFACT_DIR / "feature_discovery_response.json").write_text(
    json.dumps(feature_discovery_response_dict, indent=2, sort_keys=True),
    encoding="utf-8",
)
(FEATURE_ARTIFACT_DIR / "feature_schema.json").write_text(
    json.dumps(feature_schema, indent=2, sort_keys=True),
    encoding="utf-8",
)

bandit_X = feature_store_table.drop(columns=["feature_row_id", "event_time"])
bandit_numeric_cols = bandit_X.select_dtypes(include=["number", "bool"]).columns.tolist()
bandit_categorical_cols = [column_name for column_name in bandit_X.columns if column_name not in bandit_numeric_cols]
bandit_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, bandit_numeric_cols),
        ("cat", categorical_pipe, bandit_categorical_cols),
    ],
    remainder="drop",
)

print("Feature artifact folder:", FEATURE_ARTIFACT_DIR)
print("Feature table artifact:", feature_table_path.name)
print("MLTable metadata:", mltable_path.name)
print(f"Bandit feature table: {bandit_X.shape[1]} features after agent discovery")
feature_store_table.head(3)


In [ ]:
AZUREML_FEATURE_TABLE_ASSET_NAME = os.getenv(
    "AZUREML_FEATURE_TABLE_ASSET_NAME",
    f"{AZUREML_DATA_ASSET_NAME}-feature-table",
)
AZUREML_FEATURE_BUNDLE_ASSET_NAME = os.getenv(
    "AZUREML_FEATURE_BUNDLE_ASSET_NAME",
    f"{AZUREML_DATA_ASSET_NAME}-feature-artifacts",
)
AZUREML_FEATURE_ASSET_VERSION = os.getenv("AZUREML_FEATURE_ASSET_VERSION", AZUREML_DATA_ASSET_VERSION)


def create_or_update_versioned_data_asset(
    client: MLClient,
    *,
    name: str,
    version: str,
    asset_type: str,
    path: Path,
    description: str,
    tags: dict[str, str],
) -> Data:
    def asset(candidate_version: str) -> Data:
        return Data(
            name=name,
            version=candidate_version,
            type=asset_type,
            path=str(path),
            description=description,
            tags=tags,
        )

    try:
        return client.data.create_or_update(asset(version))
    except Exception as ex:
        fallback_version = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
        print(f"Data asset {name!r} version {version!r} was not updated: {ex}")
        print(f"Retrying with generated version {fallback_version}.")
        return client.data.create_or_update(asset(fallback_version))


feature_table_asset = None
feature_bundle_asset = None
feature_ml_client = globals().get("ml_client") or build_ml_client()

if feature_ml_client is not None:
    feature_table_asset = create_or_update_versioned_data_asset(
        feature_ml_client,
        name=AZUREML_FEATURE_TABLE_ASSET_NAME,
        version=AZUREML_FEATURE_ASSET_VERSION,
        asset_type=AssetTypes.MLTABLE,
        path=FEATURE_ARTIFACT_DIR,
        description="7-MLET curated feature table with aggregate-reviewed feature propositions.",
        tags={"stage": "curated_features", "asset_kind": "feature_table", "privacy": "agent_aggregate_only"},
    )
    feature_bundle_asset = create_or_update_versioned_data_asset(
        feature_ml_client,
        name=AZUREML_FEATURE_BUNDLE_ASSET_NAME,
        version=AZUREML_FEATURE_ASSET_VERSION,
        asset_type=AssetTypes.URI_FOLDER,
        path=FEATURE_ARTIFACT_DIR,
        description="7-MLET feature profile, discovery response, schema, and MLTable artifacts.",
        tags={"stage": "feature_discovery", "asset_kind": "feature_report_bundle", "privacy": "agent_aggregate_only"},
    )
    os.environ["AZUREML_FEATURE_TABLE_ASSET_ID"] = feature_table_asset.id
    os.environ["AZUREML_FEATURE_BUNDLE_ASSET_ID"] = feature_bundle_asset.id
    print("Registered Azure ML feature table asset:", feature_table_asset.name, feature_table_asset.version)
    print("Registered Azure ML feature bundle asset:", feature_bundle_asset.name, feature_bundle_asset.version)
else:
    print("Azure ML credentials/config not available; feature artifacts remain local:", FEATURE_ARTIFACT_DIR)


In [ ]:
FEATURE_STORE_SPEC_DIR = FEATURE_ARTIFACT_DIR / "azureml_feature_store_specs"
FEATURE_STORE_SPEC_DIR.mkdir(parents=True, exist_ok=True)

AZUREML_FEATURE_STORE_NAME = os.getenv(
    "AZUREML_FEATURE_STORE_NAME",
    PROJECT_DEFAULTS.feature_store.name,
)
AZUREML_FEATURE_STORE_RESOURCE_GROUP = os.getenv(
    "AZUREML_FEATURE_STORE_RESOURCE_GROUP",
    PROJECT_DEFAULTS.feature_store.resource_group,
)
AZUREML_FEATURE_STORE_LOCATION = os.getenv("AZUREML_FEATURE_STORE_LOCATION", PROJECT_DEFAULTS.feature_store.location)
AZUREML_FEATURE_STORE_ENTITY_NAME = os.getenv("AZUREML_FEATURE_STORE_ENTITY_NAME", "bank_marketing_customer")
AZUREML_FEATURE_SET_NAME = os.getenv("AZUREML_FEATURE_SET_NAME", "bank_marketing_customer_features")
AZUREML_FEATURE_SET_VERSION = os.getenv("AZUREML_FEATURE_SET_VERSION", AZUREML_FEATURE_ASSET_VERSION)
ENABLE_AZUREML_FEATURE_STORE_CREATE = os.getenv("ENABLE_AZUREML_FEATURE_STORE_CREATE", "false").lower() in {"1", "true", "yes"}


def write_feature_store_specs() -> dict[str, Path]:
    feature_store_yaml = FEATURE_STORE_SPEC_DIR / "feature_store.yml"
    entity_yaml = FEATURE_STORE_SPEC_DIR / "feature_store_entity.yml"
    feature_set_yaml = FEATURE_STORE_SPEC_DIR / "feature_set.yml"

    feature_store_yaml.write_text(
        "\n".join(
            [
                "$schema: https://azuremlschemas.azureedge.net/latest/featureStore.schema.json",
                f"name: {AZUREML_FEATURE_STORE_NAME}",
                f"location: {AZUREML_FEATURE_STORE_LOCATION}",
                "display_name: 7-MLET Feature Store",
                "description: Feature store for aggregate-reviewed bank marketing features.",
                "tags:",
                "  workload: 7mlet-bandits",
                "  privacy: aggregate-agent-review",
                "",
            ]
        ),
        encoding="utf-8",
    )
    entity_yaml.write_text(
        "\n".join(
            [
                "$schema: https://azuremlschemas.azureedge.net/latest/featureStoreEntity.schema.json",
                f"name: {AZUREML_FEATURE_STORE_ENTITY_NAME}",
                "version: '1'",
                "description: Synthetic customer entity for 7-MLET feature-store demo.",
                "index_columns:",
                "  - name: feature_row_id",
                "    type: integer",
                "tags:",
                "  workload: 7mlet-bandits",
                "",
            ]
        ),
        encoding="utf-8",
    )
    feature_set_yaml.write_text(
        "\n".join(
            [
                "$schema: https://azuremlschemas.azureedge.net/latest/featureset.schema.json",
                f"name: {AZUREML_FEATURE_SET_NAME}",
                f"version: '{AZUREML_FEATURE_SET_VERSION}'",
                "description: Curated bank marketing features proposed from aggregate agent review.",
                "entities:",
                f"  - name: {AZUREML_FEATURE_STORE_ENTITY_NAME}",
                "    version: '1'",
                "specification:",
                f"  path: {FEATURE_ARTIFACT_DIR.as_posix()}",
                "stage: Development",
                "tags:",
                "  workload: 7mlet-bandits",
                "  source: notebook-feature-discovery",
                "",
            ]
        ),
        encoding="utf-8",
    )
    return {"feature_store": feature_store_yaml, "entity": entity_yaml, "feature_set": feature_set_yaml}


def feature_store_cli_commands(specs: dict[str, Path]) -> list[list[str]]:
    return [
        [
            "az",
            "ml",
            "feature-store",
            "create",
            "--file",
            str(specs["feature_store"]),
            "--resource-group",
            AZUREML_FEATURE_STORE_RESOURCE_GROUP,
        ],
        [
            "az",
            "ml",
            "feature-store-entity",
            "create",
            "--file",
            str(specs["entity"]),
            "--resource-group",
            AZUREML_FEATURE_STORE_RESOURCE_GROUP,
            "--feature-store-name",
            AZUREML_FEATURE_STORE_NAME,
        ],
        [
            "az",
            "ml",
            "feature-set",
            "create",
            "--file",
            str(specs["feature_set"]),
            "--resource-group",
            AZUREML_FEATURE_STORE_RESOURCE_GROUP,
            "--feature-store-name",
            AZUREML_FEATURE_STORE_NAME,
        ],
    ]


feature_store_specs = write_feature_store_specs()
commands = feature_store_cli_commands(feature_store_specs)

print("Feature Store specs written to:", FEATURE_STORE_SPEC_DIR)
for command_parts in commands:
    print(" ".join(command_parts))

if ENABLE_AZUREML_FEATURE_STORE_CREATE:
    if shutil.which("az") is None:
        raise RuntimeError("Azure CLI is not available in this environment.")
    for command_parts in commands:
        completed = subprocess.run(command_parts, text=True, capture_output=True)
        print(completed.stdout)
        if completed.returncode != 0:
            print(completed.stderr)
            print("Feature Store creation did not complete; keeping Azure ML data assets as the runnable baseline.")
            break
    else:
        print("Feature Store creation commands completed.")
else:
    print("Feature Store creation is gated. Set ENABLE_AZUREML_FEATURE_STORE_CREATE=true to run the commands from this cell.")


## 9) Optional Classical ML Sanity Check

We train a simple logistic regression only as a sanity check that the preprocessed features carry conversion signal. This is not the bandit policy.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("logreg", LogisticRegression(max_iter=1000)),
])

clf.fit(X_train, y_train)
proba_test = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba_test)
print(f"Validation ROC-AUC (sanity check): {auc:.4f}")

## 10) Build Synthetic Contextual Bandit Layer

To evaluate online policies reproducibly, we create a synthetic environment on top of real customer contexts:
- reduce transformed feature space to compact dense context vectors
- define an offer catalog with business margins
- define hidden conversion probabilities per offer (unknown to policies)
- generate logged interactions and delayed feedback

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


def build_context_matrix(
    X_data: pd.DataFrame,
    preproc: ColumnTransformer,
    n_components: int = 12,
) -> np.ndarray:
    X_sparse = preproc.fit_transform(X_data)
    max_comp = max(2, min(n_components, X_sparse.shape[1] - 1))
    reducer = TruncatedSVD(n_components=max_comp, random_state=42)
    X_dense = reducer.fit_transform(X_sparse)
    return X_dense


offer_catalog = pd.DataFrame(
    [
        {"arm": 0, "offer_name": "No Incentive", "margin": 1.0},
        {"arm": 1, "offer_name": "Fee Discount", "margin": 1.2},
        {"arm": 2, "offer_name": "Cashback", "margin": 1.4},
        {"arm": 3, "offer_name": "Premium Bundle", "margin": 1.8},
    ]
)

contexts = build_context_matrix(bandit_X, bandit_preprocessor, n_components=12)
n_rounds, context_dim = contexts.shape
n_arms = len(offer_catalog)
margins = offer_catalog["margin"].to_numpy()

true_weights = RNG.normal(loc=0.0, scale=0.35, size=(n_arms, context_dim))
true_bias = np.array([-0.6, -0.4, -0.3, -0.2])

all_logits = contexts @ true_weights.T + true_bias
all_probs = sigmoid(all_logits)
all_expected_rewards = all_probs * margins

print(f"Bandit rounds: {n_rounds}, context dim: {context_dim}, arms: {n_arms}")
offer_catalog

In [ ]:
def generate_logged_interactions(
    contexts_arr: np.ndarray,
    probs: np.ndarray,
    margins_arr: np.ndarray,
    epsilon: float = 0.35,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n, k = probs.shape

    actions = np.zeros(n, dtype=int)
    chosen_prob = np.zeros(n, dtype=float)
    conversion = np.zeros(n, dtype=int)
    reward = np.zeros(n, dtype=float)
    delay_steps = np.zeros(n, dtype=int)

    for t in range(n):
        greedy_arm = int(np.argmax(probs[t] * margins_arr))
        if rng.random() < epsilon:
            a = int(rng.integers(0, k))
        else:
            a = greedy_arm

        p = probs[t, a]
        conv = int(rng.random() < p)
        r = conv * margins_arr[a]
        d = int(min(rng.geometric(0.35) - 1, 8))

        actions[t] = a
        chosen_prob[t] = p
        conversion[t] = conv
        reward[t] = r
        delay_steps[t] = d

    logged = pd.DataFrame({
        "round": np.arange(n),
        "action": actions,
        "true_p": chosen_prob,
        "conversion": conversion,
        "reward": reward,
        "delay_steps": delay_steps,
    })
    return logged


logged_df = generate_logged_interactions(contexts, all_probs, margins, epsilon=0.35, seed=42)
logged_df.head(10)

## 11) Policy Definitions

### Deterministic baseline
A fixed business rule: always send the same offer (`Premium Bundle`). This is simple and fully deterministic, but does not adapt to context.

### UCB intuition
Upper Confidence Bound adds optimism for uncertain arms. Arms with fewer observations get higher uncertainty bonuses, creating principled exploration.

### Thompson Sampling intuition
For Bernoulli conversion, each arm keeps a Beta posterior over conversion probability. At each round, sample one plausible conversion rate per arm and pick the arm with best sampled expected reward.

In [ ]:
@dataclass
class RoundOutcome:
    action: int
    conversion: int
    reward: float
    expected_reward: float
    optimal_expected_reward: float
    explored: int


class DeterministicBaselinePolicy:
    def __init__(self, fixed_arm: int, n_arms: int):
        self.fixed_arm = fixed_arm
        self.n_arms = n_arms

    def select_action(self, t: int) -> Tuple[int, int]:
        return self.fixed_arm, 0

    def update(self, action: int, conversion: int) -> None:
        return


class UCBPolicy:
    def __init__(self, n_arms: int, margins_arr: np.ndarray, alpha: float = 1.0):
        self.n_arms = n_arms
        self.margins = margins_arr
        self.alpha = alpha
        self.successes = np.zeros(n_arms, dtype=float)
        self.failures = np.zeros(n_arms, dtype=float)

    @property
    def counts(self) -> np.ndarray:
        return self.successes + self.failures

    def select_action(self, t: int) -> Tuple[int, int]:
        counts = self.counts
        total = np.maximum(1.0, counts.sum())
        mean = (self.successes + 1.0) / (counts + 2.0)
        bonus = self.alpha * np.sqrt(np.log(total + 1.0) / (counts + 1.0))
        ucb_score = (mean + bonus) * self.margins

        greedy = int(np.argmax(mean * self.margins))
        action = int(np.argmax(ucb_score))
        explored = int(action != greedy)
        return action, explored

    def update(self, action: int, conversion: int) -> None:
        if conversion:
            self.successes[action] += 1.0
        else:
            self.failures[action] += 1.0


class ThompsonSamplingPolicy:
    def __init__(self, n_arms: int, margins_arr: np.ndarray):
        self.n_arms = n_arms
        self.margins = margins_arr
        self.alpha = np.ones(n_arms, dtype=float)
        self.beta = np.ones(n_arms, dtype=float)

    def select_action(self, t: int) -> Tuple[int, int]:
        sampled_theta = RNG.beta(self.alpha, self.beta)
        sampled_reward = sampled_theta * self.margins

        post_mean = self.alpha / (self.alpha + self.beta)
        greedy = int(np.argmax(post_mean * self.margins))
        action = int(np.argmax(sampled_reward))
        explored = int(action != greedy)
        return action, explored

    def update(self, action: int, conversion: int) -> None:
        self.alpha[action] += conversion
        self.beta[action] += 1 - conversion

## 12) Simulation with Delayed Rewards

We simulate online decisions over customer contexts.

Delayed reward handling:
- each impression gets a delay in rounds before feedback is observable
- policy updates happen only when delayed feedback matures
- metrics are tracked by decision time

In [ ]:
def run_policy_simulation(
    policy_name: str,
    policy,
    probs: np.ndarray,
    margins_arr: np.ndarray,
    max_delay: int = 8,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n, _ = probs.shape

    pending: Dict[int, List[Tuple[int, int]]] = {}
    rows: List[RoundOutcome] = []

    for t in range(n):
        matured = pending.pop(t, [])
        for act, conv in matured:
            policy.update(act, conv)

        action, explored = policy.select_action(t)
        p = probs[t, action]
        conversion = int(rng.random() < p)
        reward = conversion * margins_arr[action]

        delay = int(min(rng.geometric(0.35) - 1, max_delay))
        update_t = t + delay
        pending.setdefault(update_t, []).append((action, conversion))

        expected_reward = p * margins_arr[action]
        optimal_expected = float(np.max(probs[t] * margins_arr))

        rows.append(
            RoundOutcome(
                action=action,
                conversion=conversion,
                reward=reward,
                expected_reward=expected_reward,
                optimal_expected_reward=optimal_expected,
                explored=explored,
            )
        )

    result = pd.DataFrame([r.__dict__ for r in rows])
    result["policy"] = policy_name
    result["round"] = np.arange(len(result))
    result["instant_regret"] = result["optimal_expected_reward"] - result["expected_reward"]
    result["cum_reward"] = result["reward"].cumsum()
    result["cum_regret"] = result["instant_regret"].cumsum()
    result["cum_conversions"] = result["conversion"].cumsum()
    result["cum_conversion_rate"] = result["cum_conversions"] / (result["round"] + 1)
    result["cum_exploration_share"] = result["explored"].cumsum() / (result["round"] + 1)
    return result


baseline_policy = DeterministicBaselinePolicy(fixed_arm=3, n_arms=n_arms)
ucb_policy = UCBPolicy(n_arms=n_arms, margins_arr=margins, alpha=1.2)
ts_policy = ThompsonSamplingPolicy(n_arms=n_arms, margins_arr=margins)

sim_baseline = run_policy_simulation("Deterministic Baseline", baseline_policy, all_probs, margins, seed=100)
sim_ucb = run_policy_simulation("UCB", ucb_policy, all_probs, margins, seed=100)
sim_ts = run_policy_simulation("Thompson Sampling", ts_policy, all_probs, margins, seed=100)

sim_all = pd.concat([sim_baseline, sim_ucb, sim_ts], ignore_index=True)
sim_all.head()

## 13) Aggregate Metrics

In [ ]:
summary = (
    sim_all.groupby("policy", as_index=False)
    .agg(
        total_reward=("reward", "sum"),
        cumulative_regret=("instant_regret", "sum"),
        conversion_rate=("conversion", "mean"),
        exploration_share=("explored", "mean"),
    )
    .sort_values(by="total_reward", ascending=False)
)
summary

## 14) Two-Way Foundry Agent Integration

This step demonstrates the agentic loop on the workload:

Raw customer rows stay in the Azure ML data layer; only aggregate policy evidence is sent to the agent.

1. The notebook sends only aggregate policy evidence to an Azure AI Foundry agent.
2. The agent validates and enriches the multi-armed strategy.
3. The notebook accepts only safe bounded recommendations and reruns a candidate policy.

If Foundry SDK configuration is not available, the same contract uses a deterministic local fallback so the pipeline remains reproducible.

In [ ]:
import json
import os
import sys

candidate_src_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "aml-foundry-integration" / "src",
    Path.cwd().parent.parent / "aml-foundry-integration" / "src",
]
for package_src in candidate_src_paths:
    if (package_src / "aml_bandits" / "foundry_bridge.py").exists():
        if str(package_src) not in sys.path:
            sys.path.insert(0, str(package_src))
        break

from aml_bandits.foundry_bridge import (
    build_aggregate_metrics,
    to_agent_payload,
    validate_and_enrich_strategy,
)

aggregate_evidence = build_aggregate_metrics(summary, current_ucb_alpha=1.2)
agent_payload = to_agent_payload(aggregate_evidence)

print("Payload privacy mode:", agent_payload["privacy"])
print("Allowed recommendations:", agent_payload["allowed_recommendations"])
print("Foundry endpoint configured:", bool(os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")))
print("Foundry strategy agent:", os.getenv("FOUNDRY_STRATEGY_AGENT_ID") or os.getenv("AZURE_AI_STRATEGY_AGENT_ID") or os.getenv("FOUNDRY_STRATEGY_AGENT_NAME") or os.getenv("AZURE_AI_STRATEGY_AGENT_NAME"))

agent_recommendation = validate_and_enrich_strategy(aggregate_evidence)

agent_review = pd.DataFrame([
    {
        "source": agent_recommendation.source,
        "validation": agent_recommendation.validation,
        "accepted": agent_recommendation.accepted,
        "recommended_ucb_alpha": agent_recommendation.recommended_ucb_alpha,
        "bounded_ucb_alpha": agent_recommendation.bounded_ucb_alpha,
        "rationale": agent_recommendation.rationale,
        "warnings": "; ".join(agent_recommendation.warnings),
    }
])
agent_review


In [ ]:
if agent_recommendation.accepted and agent_recommendation.bounded_ucb_alpha is not None:
    agent_ucb_policy = UCBPolicy(
        n_arms=n_arms,
        margins_arr=margins,
        alpha=agent_recommendation.bounded_ucb_alpha,
    )
    sim_ucb_agent = run_policy_simulation(
        "UCB (Agent Recommended)",
        agent_ucb_policy,
        all_probs,
        margins,
        seed=101,
    )
    sim_all = pd.concat([sim_all, sim_ucb_agent], ignore_index=True)
    summary = (
        sim_all.groupby("policy", as_index=False)
        .agg(
            total_reward=("reward", "sum"),
            cumulative_regret=("instant_regret", "sum"),
            conversion_rate=("conversion", "mean"),
            exploration_share=("explored", "mean"),
        )
        .sort_values(by="total_reward", ascending=False)
    )
    print(
        "Applied agent recommendation: "
        f"UCB alpha={agent_recommendation.bounded_ucb_alpha:.3f}"
    )
else:
    print("No safe agent recommendation was applied; baseline policy set remains unchanged.")

summary

## 15) Visual Comparison

Simple diagnostics:
- cumulative reward (higher is better)
- cumulative regret (lower is better)
- running conversion rate
- running exploration share

If the Foundry bridge accepted a safe recommendation, the `UCB (Agent Recommended)` candidate appears alongside the baseline policies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for policy_name, grp in sim_all.groupby("policy"):
    axes[0, 0].plot(grp["round"], grp["cum_reward"], label=policy_name)
    axes[0, 1].plot(grp["round"], grp["cum_regret"], label=policy_name)
    axes[1, 0].plot(grp["round"], grp["cum_conversion_rate"], label=policy_name)
    axes[1, 1].plot(grp["round"], grp["cum_exploration_share"], label=policy_name)

axes[0, 0].set_title("Cumulative Reward")
axes[0, 1].set_title("Cumulative Regret")
axes[1, 0].set_title("Running Conversion Rate")
axes[1, 1].set_title("Running Exploration Share")

for ax in axes.ravel():
    ax.set_xlabel("Round")
    ax.grid(alpha=0.25)

axes[0, 0].set_ylabel("Reward")
axes[0, 1].set_ylabel("Regret")
axes[1, 0].set_ylabel("Conversion Rate")
axes[1, 1].set_ylabel("Exploration Share")

axes[0, 0].legend(loc="best")
axes[0, 1].legend(loc="best")
axes[1, 0].legend(loc="best")
axes[1, 1].legend(loc="best")

plt.tight_layout()
plt.show()

## 16) Findings and Limitations

### Findings
- Adaptive policies (UCB and Thompson Sampling) typically improve reward over a fixed deterministic baseline.
- UCB tends to be systematic in exploration through uncertainty bonuses.
- Thompson Sampling often balances exploration and exploitation naturally via posterior sampling.
- Delayed rewards slow learning because policy updates do not happen immediately after each action.
- The Foundry agent bridge demonstrates a two-way integration pattern: aggregate evidence leaves the pipeline, bounded strategy guidance returns, and the notebook reruns a candidate policy.

### Limitations
- Reward model is synthetic (contextual but simulated), not direct production uplift.
- Only Bernoulli conversion is modeled; no long-term customer value or churn effects.
- No propensity correction or off-policy evaluation estimator is applied in this baseline notebook.
- Offer economics are simplified to scalar margins; real campaigns may need budget and fairness constraints.
- Agent guidance is intentionally constrained to aggregate-only evidence and bounded UCB alpha changes; broader strategy changes should pass separate review gates.

### Next Steps for Datathon
- Add contextual bandits with linear models (LinUCB / contextual Thompson).
- Add policy constraints (max discount budget, segment-level caps).
- Evaluate with offline estimators (IPS/DR) before online deployment.
- Register the Foundry agent response and accepted recommendation as experiment artifacts in Azure ML.